# Train LiDAR / DG-VDSR

This notebook wires together:
- `model_anchor.py`
- `dataloader_anchor.py`
- `Hann_merge`
- training / validation / checkpoint saving
- progress bars and epoch metrics


In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, json, math, time, gc, shutil
from pathlib import Path

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from dataloader_anchor import create_dataloaders
from model_anchor import DistanceGatedGeoVDSR, DistanceGatedTopoLoss
from hann_merge import HannStreamMerger

# ── Detect environment ────────────────────────────────────────────────────
IN_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
IN_LOCAL  = not IN_COLAB
print(f'Environment: {"Colab" if IN_COLAB else "Local"}')

# =============================================================================
# ── DEFINE DEVICE HERE ───────────────────────────────────────────────────────
# =============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
torch.backends.cudnn.benchmark = True

# =============================================================================
# LOCAL WORKSTATION (P4000)
# Dataset is expected at: <project_dir>/Dataset/
# Scripts are expected at: <project_dir>/  (same folder as this notebook)
# =============================================================================
if IN_LOCAL:
    PROJECT_DIR = Path(r'D:\Vaibhav\vdsr-atl08')   # <-- edit if different
    DATASET_DIR = PROJECT_DIR / 'Dataset'

    if not DATASET_DIR.exists():
        raise FileNotFoundError(
            f'Dataset not found at {DATASET_DIR}.\n'
            f'Download it and place it at {DATASET_DIR}, or update PROJECT_DIR above.'
        )

    # Add project dir to path so scripts can be imported directly
    if str(PROJECT_DIR) not in sys.path:
        sys.path.insert(0, str(PROJECT_DIR))
    print(f'Dataset found: {DATASET_DIR}')
    print(f'Scripts:       {PROJECT_DIR}')

# =============================================================================
# GOOGLE COLAB (T4)
# Mounts Drive, copies dataset to fast local SSD (/content/dataset)
# =============================================================================
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_DATASET = Path('/content/drive/MyDrive/dem-lidar/Dataset')
    LOCAL_DATASET = Path('/content/dataset')

    if not LOCAL_DATASET.exists():
        if DRIVE_DATASET.exists():
            print('Copying dataset from Drive to local SSD (faster I/O)...')
            shutil.copytree(str(DRIVE_DATASET), str(LOCAL_DATASET))
            print('Done.')
        else:
            raise FileNotFoundError(
                f'Dataset not found at {DRIVE_DATASET}.\n'
                f'Upload your dataset to Google Drive at that path first.'
            )
    else:
        print(f'Local dataset already exists: {LOCAL_DATASET}')

    # Scripts: sourced from Drive (uploaded by setup_colab.ipynb)
    SCRIPT_DIR = Path('/content')
    if str(SCRIPT_DIR) not in sys.path:
        sys.path.insert(0, str(SCRIPT_DIR))


Environment: Local
Using device: cuda
Dataset found: D:\Vaibhav\vdsr-atl08\Dataset
Scripts:       D:\Vaibhav\vdsr-atl08


In [2]:
from pathlib import Path

# -----------------------------------------------------------------------------
# Paths
# BASE_DIR and CHECKPOINT_DIR are set automatically from Cell 1 variables.
# -----------------------------------------------------------------------------

# ── P4000 / local workstation ──────────────────────────────────────────────
# BASE_DIR and CHECKPOINT_DIR are derived from PROJECT_DIR set in Cell 1.
# No edits needed here if you set PROJECT_DIR correctly in Cell 1.
if IN_LOCAL:
    BASE_DIR       = str(DATASET_DIR)   # = PROJECT_DIR / 'Dataset'
    CHECKPOINT_DIR = PROJECT_DIR / 'checkpoints_dg_vdsr'

# ── Colab / T4 ─────────────────────────────────────────────────────────────
if IN_COLAB:
    BASE_DIR       = str(LOCAL_DATASET)  # = /content/dataset (fast local SSD)
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/checkpoints_dg_vdsr')

TRAIN_DIRS = [
    f"{BASE_DIR}/Kl/tensors/train",
    f"{BASE_DIR}/SG/tensors/train",
]
VAL_DIRS = [
    f"{BASE_DIR}/Kl/tensors/validation_contiguous",
    f"{BASE_DIR}/SG/tensors/validation_contiguous",
]
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAME = "dg_vdsr_lidar"

# -----------------------------------------------------------------------------
# Training hyperparameters
# -----------------------------------------------------------------------------
EPOCHS = 500
BATCH_SIZE = 16        # P4000 FP32: ~1.3 GB activations; try 32 if no OOM
# BATCH_SIZE = 64        # T4 AMP (FP16 halves activation memory)
LR = 1e-4
WEIGHT_DECAY = 5e-5 * (BATCH_SIZE//16)
GRAD_CLIP_NORM = 1.0
WARMUP_STEPS = (12 * 1649 // BATCH_SIZE) + BATCH_SIZE  # Step-based linear warmup (~11 epochs at 26 batches/epoch)
BASE_LR_STREAM_A = LR 
BASE_LR_STREAM_B = LR

TOPO_LAYERS = 18
FEATURES = 64

# Loss weights
LOSS_ALPHA      = 1.0      # Base elevation (optical, halo-masked)
LOSS_BETA       = 1.5      # Slope (supervised outside / smooth inside buffer)
LOSS_GAMMA      = 0.5      # Curvature (kept low: resists terrain flex the most)
LOSS_LAMBDA_PIN = 1.0      # Starting value; ramped to 5.0 over first 15 epochs
PIN_BETA        = 1.0      # SmoothL1 knee in metres (robust to +-2.48m scatter)
DECAY_RADIUS    = 15.0     # Halo decay in metres (~one ATL08 along-track spacing)
BUFFER_SIZE     = 3        # Buffer ring in PIXELS (3px x 10m = 30m ring)

# Lambda_pin warm-up schedule
LAMBDA_PIN_MAX   = 5.0
LAMBDA_PIN_START = 1.0
WARMUP_EPOCHS    = 15

# Dataloader settings
TRAIN_CROP = 128
VAL_CROP = 256
VAL_OVERLAP = 192
# P4000 workstation: 48-thread Xeon + 64 GB RAM → 8 workers is comfortable
NUM_WORKERS = 8
# NUM_WORKERS = 4        # Colab T4 (fewer CPUs available)
# NUM_WORKERS = 0        # Windows fallback if multiprocessing crashes

# Inference-only (no grads stored); peak ≈ 2x one feature map in memory
VAL_PATCH_BATCH = 32      # P4000 FP32 256×256: ~2 GB peak VRAM
# VAL_PATCH_BATCH = 128       # T4 AMP

# Training control
EARLY_STOPPING_PATIENCE = 500

# LR Scheduler
SCHEDULER_PATIENCE = 15
SCHEDULER_FACTOR   = 0.5
SCHEDULER_COOLDOWN = 2
SCHEDULER_MIN_LR   = 1e-6
BEST_METRIC_NAME = "composite"

training_config = {
    'run_name': RUN_NAME,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'base_lr_stream_a': BASE_LR_STREAM_A,
    'base_lr_stream_b': BASE_LR_STREAM_B,
    'warmup_steps': WARMUP_STEPS,
    'weight_decay': WEIGHT_DECAY,
    'grad_clip_norm': GRAD_CLIP_NORM,
    'topo_layers': TOPO_LAYERS,
    'features': FEATURES,
    'loss_alpha': LOSS_ALPHA,
    'loss_beta': LOSS_BETA,
    'loss_gamma': LOSS_GAMMA,
    'loss_lambda_pin': LOSS_LAMBDA_PIN,
    'lambda_pin_max': LAMBDA_PIN_MAX,
    'lambda_pin_start': LAMBDA_PIN_START,
    'warmup_epochs': WARMUP_EPOCHS,
    'pin_beta': PIN_BETA,
    'decay_radius': DECAY_RADIUS,
    'buffer_size': BUFFER_SIZE,
    'scheduler_patience': SCHEDULER_PATIENCE,
    'scheduler_factor': SCHEDULER_FACTOR,
    'scheduler_cooldown': SCHEDULER_COOLDOWN,
    'scheduler_min_lr': SCHEDULER_MIN_LR,
    'train_crop': TRAIN_CROP,
    'val_crop': VAL_CROP,
    'val_overlap': VAL_OVERLAP,
    'val_patch_batch': VAL_PATCH_BATCH,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'best_metric_name': BEST_METRIC_NAME,
    'train_dirs': TRAIN_DIRS,
    'val_dirs': VAL_DIRS,
}

print(json.dumps(training_config, indent=2))

{
  "run_name": "dg_vdsr_lidar",
  "epochs": 500,
  "batch_size": 16,
  "lr": 0.0001,
  "base_lr_stream_a": 0.0001,
  "base_lr_stream_b": 0.0001,
  "warmup_steps": 1252,
  "weight_decay": 5e-05,
  "grad_clip_norm": 1.0,
  "topo_layers": 18,
  "features": 64,
  "loss_alpha": 1.0,
  "loss_beta": 1.5,
  "loss_gamma": 0.5,
  "loss_lambda_pin": 1.0,
  "lambda_pin_max": 5.0,
  "lambda_pin_start": 1.0,
  "warmup_epochs": 15,
  "pin_beta": 1.0,
  "decay_radius": 15.0,
  "buffer_size": 3,
  "scheduler_patience": 15,
  "scheduler_factor": 0.5,
  "scheduler_cooldown": 2,
  "scheduler_min_lr": 1e-06,
  "train_crop": 128,
  "val_crop": 256,
  "val_overlap": 192,
  "val_patch_batch": 32,
  "early_stopping_patience": 500,
  "best_metric_name": "composite",
  "train_dirs": [
    "D:\\Vaibhav\\vdsr-atl08\\Dataset/Kl/tensors/train",
    "D:\\Vaibhav\\vdsr-atl08\\Dataset/SG/tensors/train"
  ],
  "val_dirs": [
    "D:\\Vaibhav\\vdsr-atl08\\Dataset/Kl/tensors/validation_contiguous",
    "D:\\Vaibhav\\vdsr-

In [3]:
# -----------------------------------------------------------------------------
# Train / validation helpers
# -----------------------------------------------------------------------------
def compute_rmse(pred, gt):
    return torch.sqrt(torch.mean((pred - gt) ** 2))

def compute_psnr(pred, gt, eps=1e-8):
    mse = torch.mean((pred - gt) ** 2)
    data_range = torch.clamp(gt.max() - gt.min(), min=eps)
    if mse <= eps:
        return torch.tensor(float('inf'), device=pred.device)
    return 20.0 * torch.log10(data_range) - 10.0 * torch.log10(torch.clamp(mse, min=eps))

def safe_conv(tensor, kernel):
    padded = F.pad(tensor, (1, 1, 1, 1), mode='replicate')
    return F.conv2d(padded, kernel)

sobel_x = torch.tensor(
    [[-1, 0, 1],
     [-2, 0, 2],
     [-1, 0, 1]],
    dtype=torch.float32
).view(1, 1, 3, 3) / 8.0

sobel_y = torch.tensor(
    [[-1, -2, -1],
     [0, 0, 0],
     [1, 2, 1]],
    dtype=torch.float32
).view(1, 1, 3, 3) / 8.0

laplacian = torch.tensor(
    [[0, 1, 0],
     [1, -4, 1],
     [0, 1, 0]],
    dtype=torch.float32
).view(1, 1, 3, 3)
def get_warmup_lr(step, base_lr, warmup_steps):
    if step >= warmup_steps:
        return base_lr
    return base_lr * (step + 1) / warmup_steps
    
def save_checkpoint(
    epoch, model, optimizer_a, optimizer_b, scheduler_a, scheduler_b,
    best_metric, best_epoch, train_history, checkpoint_dir, training_config, is_best=False
):
    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_a_state_dict': optimizer_a.state_dict(),
        'optimizer_b_state_dict': optimizer_b.state_dict(),
        'scheduler_a_state_dict': scheduler_a.state_dict() if scheduler_a is not None else None,
        'scheduler_b_state_dict': scheduler_b.state_dict() if scheduler_b is not None else None,
        'best_metric': best_metric,
        'best_epoch': best_epoch,
        'best_metric_name': training_config.get('best_metric_name', 'val_slope_rmse'),
        'train_history': train_history,
        'training_config': training_config,
        'torch_version': torch.__version__,
    }

    filename = checkpoint_dir / f"{RUN_NAME}_epoch_{epoch:03d}.pt"
    torch.save(ckpt, filename)

    meta_path = checkpoint_dir / f"{RUN_NAME}_epoch_{epoch:03d}_meta.json"
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump({
            'epoch': epoch,
            'best_metric': best_metric,
            'best_epoch': best_epoch,
            'training_config': training_config,
            'torch_version': torch.__version__,
        }, f, indent=2)

    if is_best:
        best_path = checkpoint_dir / f"{RUN_NAME}_best.pt"
        torch.save(ckpt, best_path)

    return filename


def train_one_epoch(
    model, criterion, optimizer_a, optimizer_b, train_loader, device, scaler, 
    grad_clip_norm=1.0, epoch=0, warmup_steps=0, base_lrs=None
):
    model.train()

    running = {
        'total': 0.0, 'base': 0.0, 'slope': 0.0,
        'curve': 0.0, 'pin': 0.0, 'mae': 0.0, 'rmse': 0.0, 'anchor_mae': 0.0,
        'grad_norm_a': 0.0, 'grad_norm_b': 0.0,
    }
    n_batches = 0

    pbar = tqdm(train_loader, desc='Train', leave=False, dynamic_ncols=True)

    for batch_idx, batch in enumerate(pbar):
        dem_bic = batch['dem_bic'].to(device, non_blocking=True, dtype=torch.float32)
        lidar_delta = batch['lidar_delta'].to(device, non_blocking=True, dtype=torch.float32)
        mask = batch['mask'].to(device, non_blocking=True, dtype=torch.float32)
        dist_map = batch['dist_map'].to(device, non_blocking=True, dtype=torch.float32)
        gt_dem = batch['gt_dem'].to(device, non_blocking=True, dtype=torch.float32)

        lidar_raw = dem_bic + lidar_delta

        # Zero both optimizers
        optimizer_a.zero_grad(set_to_none=True)
        optimizer_b.zero_grad(set_to_none=True)

        pred_dem, alpha, r_topo, r_anchor = model(dem_bic, lidar_delta, mask, dist_map)
        loss_dict = criterion(pred_dem, gt_dem, lidar_raw, mask, dist_map)

        if not torch.isfinite(loss_dict['total']):
            tqdm.write(f"WARNING: non-finite loss at epoch {epoch}, batch {batch_idx} — skipping")
            optimizer_a.zero_grad(set_to_none=True)
            optimizer_b.zero_grad(set_to_none=True)
            continue
        
        loss_dict['total'].backward()

        if grad_clip_norm is not None and grad_clip_norm > 0:
            grad_norm_a = torch.nn.utils.clip_grad_norm_(model.stream_a.parameters(), grad_clip_norm)
            grad_norm_b = torch.nn.utils.clip_grad_norm_(model.stream_b.parameters(), grad_clip_norm)
            running['grad_norm_a'] += float(grad_norm_a)
            running['grad_norm_b'] += float(grad_norm_b)
            
        if warmup_steps > 0 and base_lrs is not None:
            global_step = (epoch - 1) * len(train_loader) + batch_idx
            if global_step < warmup_steps:
                optimizer_a.param_groups[0]['lr'] = get_warmup_lr(global_step, base_lrs[0], warmup_steps)
                optimizer_b.param_groups[0]['lr'] = get_warmup_lr(global_step, base_lrs[1], warmup_steps)
            
        # Step both optimizers
        optimizer_a.step()
        optimizer_b.step()

        # ... (rest of train_one_epoch remains the same) ...
        # (Make sure to update the pbar.set_postfix 'lr' to just show one, e.g., optimizer_a.param_groups[0]['lr'])

        # ── AMP block (T4 / Ampere+): comment FP32 block above, uncomment below ──
        # with torch.amp.autocast('cuda'):
        #     pred_dem, alpha, r_topo, r_anchor = model(dem_bic, lidar_delta, mask, dist_map)
        #     loss_dict = criterion(pred_dem, gt_dem, lidar_raw, mask, dist_map)
        # scaler.scale(loss_dict['total']).backward()
        # if grad_clip_norm is not None and grad_clip_norm > 0:
        #     scaler.unscale_(optimizer)
        #     if epoch <= LOG_GRAD_EPOCHS:
        #         grad_norm_a = torch.nn.utils.clip_grad_norm_(model.stream_a.parameters(), grad_clip_norm)
        #         grad_norm_b = torch.nn.utils.clip_grad_norm_(model.stream_b.parameters(), grad_clip_norm)
        #         running['grad_norm_a'] += float(grad_norm_a)
        #         running['grad_norm_b'] += float(grad_norm_b)
        #     else:
        #         torch.nn.utils.clip_grad_norm_(model.stream_a.parameters(), grad_clip_norm)
        #         torch.nn.utils.clip_grad_norm_(model.stream_b.parameters(), grad_clip_norm)
        # scaler.step(optimizer)
        # scaler.update()

        # Metrics calculation remains in float32 for accuracy
        with torch.no_grad():
            batch_mae = F.l1_loss(pred_dem, gt_dem)
            batch_rmse = compute_rmse(pred_dem, gt_dem)

            # Calculate error strictly at the LiDAR locations
            mask_sum = mask.sum()
            if mask_sum > 0:
                batch_anchor_mae = (mask * torch.abs(pred_dem - lidar_raw)).sum() / mask_sum
            else:
                batch_anchor_mae = torch.tensor(0.0, device=device)


        running['total'] += float(loss_dict['total'].item())
        running['base'] += float(loss_dict['base'].item())
        running['slope'] += float(loss_dict['slope'].item())
        running['curve'] += float(loss_dict['curve'].item())
        running['pin'] += float(loss_dict['pin'].item())
        running['mae'] += float(batch_mae.item())
        running['rmse'] += float(batch_rmse.item())
        running['anchor_mae'] += float(batch_anchor_mae.item())
        n_batches += 1

        pbar.set_postfix({
            'loss': f"{loss_dict['total'].item():.4f}",
            'mae': f"{batch_mae.item():.4f}",
            'rmse': f"{batch_rmse.item():.4f}",
            'lr_A': f"{optimizer_a.param_groups[0]['lr']:.2e}",
        })

    for k in running:
        running[k] /= max(n_batches, 1)

    return running

@torch.inference_mode()
def validate_one_epoch(model, criterion, val_loader, device, val_patch_batch=64):
    model.eval()

    # Keep metric kernels on CPU - no need to use GPU for full-DEM convolutions
    sobel_x_cpu = sobel_x.cpu()
    sobel_y_cpu = sobel_y.cpu()
    laplacian_cpu = laplacian.cpu()

    total_loss = 0.0
    total_mae = 0.0
    total_rmse = 0.0
    total_psnr = 0.0
    total_slope_rmse = 0.0
    total_curve_rmse = 0.0
    total_anchor_mae = 0.0
    total_rtopo_far = 0.0
    n_samples = 0

    outer = tqdm(val_loader, desc='Validate files', leave=False, dynamic_ncols=True)

    for sample_idx, batch in enumerate(outer, start=1):
        # for T4
        # dem_bic_all = batch['dem_bic'].squeeze(0).to(device, dtype=torch.float32)
        # lidar_delta_all = batch['lidar_delta'].squeeze(0).to(device, dtype=torch.float32)
        # mask_all = batch['mask'].squeeze(0).to(device, dtype=torch.float32)
        # dist_map_all = batch['dist_map'].squeeze(0).to(device, dtype=torch.float32)
        # gt_dem_all = batch['gt_dem'].squeeze(0).to(device, dtype=torch.float32)

        # for quadro p4000
        # Keep these on the CPU and convert to fp32
        dem_bic_all = batch['dem_bic'].squeeze(0).float()
        lidar_delta_all = batch['lidar_delta'].squeeze(0).float()
        mask_all = batch['mask'].squeeze(0).float()
        dist_map_all = batch['dist_map'].squeeze(0).float()
        gt_dem_all = batch['gt_dem'].squeeze(0).float()
        

        patch_mean_all = batch['patch_mean'].squeeze(0)
        coords_all = batch['coords'].squeeze(0)
        canvas_shape = batch['canvas_shape'].squeeze(0).tolist()
        pad = batch["pad"].item()

        original_shape = (
            batch["original_shape"]
            .squeeze(0)
            .tolist()
        )

        # Keep gt_canvas on CPU - GPU not needed for final metric computation
        gt_canvas = (
            batch['gt_canvas_full']
            .squeeze(0)
            .float()
            .unsqueeze(0)
            .unsqueeze(0)
        )

        # Accumulate on CPU to avoid holding a large canvas tensor on GPU
        merger = HannStreamMerger(
            canvas_shape=canvas_shape,
            patch_size=256,
            device="cpu",
            pad=pad,
            original_shape=original_shape,
        )

        patch_preds = []
        patch_count = dem_bic_all.shape[0]
        n_patch_batches = math.ceil(patch_count / val_patch_batch)

        inner = tqdm(
            range(n_patch_batches),
            desc=f'Patches {sample_idx}/{len(val_loader)}',
            leave=False,
            dynamic_ncols=True
        )

        running_val_loss = {'total': 0.0}
        n_val_batches = 0
        file_anchor_mae_sum = 0.0
        file_mask_sum = mask_all.sum()
        file_rtopo_far_sum = 0.0  # <--- ADD THIS
        file_far_mask_sum = 0.0   # <--- ADD THIS

        # for batch_idx in inner:
        #     start = batch_idx * val_patch_batch
        #     end = min(start + val_patch_batch, patch_count)

        #     dem_bic = dem_bic_all[start:end]
        #     lidar_delta = lidar_delta_all[start:end]
        #     mask = mask_all[start:end]
        #     dist_map = dist_map_all[start:end]
            
        #     with torch.amp.autocast('cuda'):
        #         pred_dem, alpha, r_topo, r_anchor = model(
        #             dem_bic,
        #             lidar_delta,
        #             mask,
        #             dist_map
        #         )
                
        #         # Compute loss per mini-batch to prevent VRAM explosion
        #         lidar_raw_batch = dem_bic + lidar_delta
        #         gt_dem_batch = gt_dem_all[start:end]
        #         loss_dict = criterion(pred_dem, gt_dem_batch, lidar_raw_batch, mask, dist_map)
        #         running_val_loss['total'] += float(loss_dict['total'].item())

        for batch_idx in inner:
            start = batch_idx * val_patch_batch
            end = min(start + val_patch_batch, patch_count)
        
            # Move ONLY the current 32/64 patches to the GPU right when you need them
            dem_bic = dem_bic_all[start:end].to(device, non_blocking=True, dtype=torch.float32)
            lidar_delta = lidar_delta_all[start:end].to(device, non_blocking=True, dtype=torch.float32)
            mask = mask_all[start:end].to(device, non_blocking=True, dtype=torch.float32)
            dist_map = dist_map_all[start:end].to(device, non_blocking=True, dtype=torch.float32)
            gt_dem_batch = gt_dem_all[start:end].to(device, non_blocking=True, dtype=torch.float32)
            
            # with torch.amp.autocast('cuda'):
            #     pred_dem, alpha, r_topo, r_anchor = model(
            #         dem_bic, lidar_delta, mask, dist_map
            #     )
                
            #     lidar_raw_batch = dem_bic + lidar_delta
                
            #     # Loss computation
            #     loss_dict = criterion(pred_dem, gt_dem_batch, lidar_raw_batch, mask, dist_map)
            #     running_val_loss['total'] += float(loss_dict['total'].item())
                
            #     if mask.sum() > 0:
            #         file_anchor_mae_sum += float((mask * torch.abs(pred_dem - lidar_raw_batch)).sum().item())
            pred_dem, alpha, r_topo, r_anchor = model(dem_bic, lidar_delta, mask, dist_map)
            lidar_raw_batch = dem_bic + lidar_delta
            
            # Loss computation
            loss_dict = criterion(pred_dem, gt_dem_batch, lidar_raw_batch, mask, dist_map)
            running_val_loss['total'] += float(loss_dict['total'].item())
            
            if mask.sum() > 0:
                file_anchor_mae_sum += float((mask * torch.abs(pred_dem - lidar_raw_batch)).sum().item())

            buffer_thresh = criterion.buffer_metres if hasattr(criterion, 'buffer_metres') else 30.0
            far_mask = (dist_map > buffer_thresh).float()
            
            file_far_mask_sum += float(far_mask.sum().item())
            if far_mask.sum() > 0:
                file_rtopo_far_sum += float((far_mask * torch.abs(r_topo)).sum().item())
            
            n_val_batches += 1

            # Remove patch_preds.append(pred_dem) to free GPU memory immediately!

            merger.add_batch(
                pred_dem.detach().cpu(),
                patch_mean_all[start:end],
                coords_all[start:end].cpu()
            )

            inner.set_postfix({
                'batch': f'{batch_idx + 1}/{n_patch_batches}',
            })

            # print(f"    epoch file={sample_idx} batch={batch_idx} "
            #           f"alloc={torch.cuda.memory_allocated()/1e9:.2f}GB "
            #           f"reserved={torch.cuda.memory_reserved()/1e9:.2f}GB")

        val_loss = {'total': running_val_loss['total'] / max(n_val_batches, 1)}
        
        if file_mask_sum > 0:
            val_anchor_mae = file_anchor_mae_sum / float(file_mask_sum.item())
        else:
            val_anchor_mae = 0.0
            
        total_anchor_mae += val_anchor_mae

        if file_far_mask_sum > 0:
            val_rtopo_far = file_rtopo_far_sum / file_far_mask_sum
        else:
            val_rtopo_far = 0.0
            
        total_rtopo_far += val_rtopo_far

        # Get merged DEM on CPU - all metric math stays off the GPU
        merged_pred = merger.get_final_dem(as_tensor=True).unsqueeze(0).unsqueeze(0)  # CPU

        # Explicit cleanup: free GPU memory BEFORE starting next file
        del dem_bic_all, lidar_delta_all, mask_all, dist_map_all, gt_dem_all
        del merger
        # gc.collect()
        # torch.cuda.empty_cache()

        # All metric computation on CPU (gt_canvas is already CPU)
        mae = F.l1_loss(merged_pred, gt_canvas)
        rmse = compute_rmse(merged_pred, gt_canvas)
        psnr = compute_psnr(merged_pred, gt_canvas)

        pred_dx = safe_conv(merged_pred, sobel_x_cpu)
        pred_dy = safe_conv(merged_pred, sobel_y_cpu)
        gt_dx = safe_conv(gt_canvas, sobel_x_cpu)
        gt_dy = safe_conv(gt_canvas, sobel_y_cpu)

        pred_slope = torch.sqrt(pred_dx ** 2 + pred_dy ** 2)
        gt_slope = torch.sqrt(gt_dx ** 2 + gt_dy ** 2)
        slope_rmse = compute_rmse(pred_slope, gt_slope)

        pred_curve = safe_conv(merged_pred, laplacian_cpu)
        gt_curve = safe_conv(gt_canvas, laplacian_cpu)
        curve_rmse = compute_rmse(pred_curve, gt_curve)

        del merged_pred, gt_canvas

        total_loss += val_loss['total']
        total_mae += float(mae.item())
        total_rmse += float(rmse.item())
        total_psnr += float(psnr.item())
        total_slope_rmse += float(slope_rmse.item())
        total_curve_rmse += float(curve_rmse.item())
        n_samples += 1

        outer.set_postfix({
            'mae': f'{mae.item():.3f}',
            'rmse': f'{rmse.item():.3f}',
            'psnr': f'{psnr.item():.2f}',
            'slope': f'{slope_rmse.item():.3f}',
            'curve': f'{curve_rmse.item():.3f}',
        })

    return {
        'val_total': total_loss / max(n_samples, 1),
        'val_mae': total_mae / max(n_samples, 1),
        'val_rmse': total_rmse / max(n_samples, 1),
        'val_psnr': total_psnr / max(n_samples, 1),
        'val_slope_rmse': total_slope_rmse / max(n_samples, 1),
        'val_curve_rmse': total_curve_rmse / max(n_samples, 1),
        'val_anchor_mae': total_anchor_mae / max(n_samples, 1),
        'val_rtopo_far': total_rtopo_far / max(n_samples, 1),
    }

In [ ]:
def main():
    train_loader, val_loader = create_dataloaders(
        TRAIN_DIRS,
        VAL_DIRS,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        prefetch_factor=4 if NUM_WORKERS > 0 else None,
        pin_memory=True,   # P4000 supports pinned memory; faster CPU->GPU transfer
        # pin_memory=(NUM_WORKERS > 0),  # use this if running NUM_WORKERS=0
    )

    model = DistanceGatedGeoVDSR(topo_layers=TOPO_LAYERS, features=FEATURES).to(device)
    criterion = DistanceGatedTopoLoss(
        alpha        = LOSS_ALPHA,
        beta         = LOSS_BETA,
        gamma        = LOSS_GAMMA,
        lambda_pin   = LOSS_LAMBDA_PIN,   # Will be overwritten each epoch by warm-up
        pin_beta     = PIN_BETA,
        decay_radius = DECAY_RADIUS,
        buffer_size  = BUFFER_SIZE,
    ).to(device)

    optimizer_a = torch.optim.Adam(model.stream_a.parameters(), lr=BASE_LR_STREAM_A, weight_decay=WEIGHT_DECAY)
    optimizer_b = torch.optim.Adam(model.stream_b.parameters(), lr=BASE_LR_STREAM_B, weight_decay=WEIGHT_DECAY)

    scheduler_a = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_a, mode='min', factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE,
        cooldown=SCHEDULER_COOLDOWN, min_lr=SCHEDULER_MIN_LR)
    scheduler_b = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_b, mode='min', factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE,
        cooldown=SCHEDULER_COOLDOWN, min_lr=SCHEDULER_MIN_LR)

    # scaler = torch.amp.GradScaler('cuda')   # Uncomment for AMP on T4/Ampere+
    scaler = torch.amp.GradScaler('cuda', enabled=False)  # disabled: FP32-only (P4000)


    # ── Checkpoint Resume ─────────────────────────────────────────────────────
    # Set RESUME_CHECKPOINT to the path of a .pt file to continue training.
    # Leave as None to start from scratch.
    RESUME_CHECKPOINT = CHECKPOINT_DIR / 'dg_vdsr_lidar_epoch_031.pt'
   # RESUME_CHECKPOINT = None

    # 1. Set fresh-start defaults
    start_epoch    = 1
    best_metric    = float('inf')
    best_composite = float('inf')
    best_epoch     = 0
    stale_epochs   = 0
    train_history  = []

    # 2. Overwrite defaults if resuming from a checkpoint
    if RESUME_CHECKPOINT is not None:
        ckpt = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        
        # Load dual optimizers and schedulers
        if 'optimizer_a_state_dict' in ckpt:
            optimizer_a.load_state_dict(ckpt['optimizer_a_state_dict'])
            optimizer_b.load_state_dict(ckpt['optimizer_b_state_dict'])
            if ckpt.get('scheduler_a_state_dict') is not None:
                scheduler_a.load_state_dict(ckpt['scheduler_a_state_dict'])
                scheduler_b.load_state_dict(ckpt['scheduler_b_state_dict'])
                scheduler_a.cooldown = SCHEDULER_COOLDOWN
                scheduler_b.cooldown = SCHEDULER_COOLDOWN
                scheduler_a.min_lrs = [SCHEDULER_MIN_LR] * len(scheduler_a.min_lrs)
                scheduler_b.min_lrs = [SCHEDULER_MIN_LR] * len(scheduler_b.min_lrs)
        else:
            print("Warning: Loading an older single-optimizer checkpoint. Optimizer states reset.")

        # Restore tracking variables to protect the best model and history logs
        start_epoch    = ckpt['epoch'] + 1
        best_metric    = ckpt.get('best_metric', float('inf'))
        best_composite = ckpt.get('best_metric', float('inf'))
        best_epoch     = ckpt.get('best_epoch', 0)
        train_history  = ckpt.get('train_history', [])
        
        print(f'Resumed from epoch {ckpt["epoch"]} | best_composite={best_composite:.4f} @ ep {best_epoch}')
    else:
        print('Starting from scratch.')

    print(f'Train files: {len(train_loader.dataset)}')
    print(f'Val files:   {len(val_loader.dataset)}')

    epoch_bar = tqdm(range(start_epoch, EPOCHS + 1), desc='Epochs', dynamic_ncols=True)

    for epoch in epoch_bar:
        t0 = time.time()

        # Lambda_pin warm-up: ramp from LAMBDA_PIN_START → LAMBDA_PIN_MAX over WARMUP_EPOCHS
        current_lambda_pin = min(
            LAMBDA_PIN_MAX,
            LAMBDA_PIN_START + (LAMBDA_PIN_MAX - LAMBDA_PIN_START) * ((epoch - 1) / WARMUP_EPOCHS)
        )
        criterion.lambda_pin = current_lambda_pin

        train_stats = train_one_epoch(
            model=model,
            criterion=criterion,
            optimizer_a=optimizer_a,  
            optimizer_b=optimizer_b, 
            train_loader=train_loader,
            device=device,
            scaler=scaler,
            grad_clip_norm=GRAD_CLIP_NORM,
            epoch=epoch,
            warmup_steps=WARMUP_STEPS,
            base_lrs=[BASE_LR_STREAM_A, BASE_LR_STREAM_B],
        )

        val_stats = validate_one_epoch(
            model=model,
            criterion=criterion,
            val_loader=val_loader,
            device=device,
            val_patch_batch=VAL_PATCH_BATCH,
        )

        # Combined score: LiDAR accuracy + terrain smoothness
        composite = val_stats['val_slope_rmse'] + val_stats['val_anchor_mae']

        if epoch > WARMUP_EPOCHS:
            # Stream A watches Slope RMSE
            scheduler_a.step(val_stats['val_slope_rmse'])
            # Stream B watches Anchor MAE
            scheduler_b.step(val_stats['val_anchor_mae'])

        lr_a = optimizer_a.param_groups[0]['lr']
        lr_b = optimizer_b.param_groups[0]['lr']
        elapsed = time.time() - t0
            

        row = {
            'epoch': epoch,
            'lr_a': lr_a,
            'lr_b': lr_b,
            'train_total': train_stats['total'],
            'train_base': train_stats['base'],
            'train_slope': train_stats['slope'],
            'train_curve': train_stats['curve'],
            'train_pin': train_stats['pin'],
            'train_mae': train_stats['mae'],
            'train_rmse': train_stats['rmse'],
            'val_total': val_stats['val_total'],
            'val_mae': val_stats['val_mae'],
            'val_rmse': val_stats['val_rmse'],
            'val_psnr': val_stats['val_psnr'],
            'val_slope_rmse': val_stats['val_slope_rmse'],
            'val_curve_rmse': val_stats['val_curve_rmse'],
            'val_anchor_mae': val_stats['val_anchor_mae'],
            'val_rtopo_far': val_stats['val_rtopo_far'],
            'val_composite':  val_stats['val_slope_rmse'] + val_stats['val_anchor_mae'],
            'time_sec': elapsed,
        }
        train_history.append(row)

        is_best = composite < best_composite
        if is_best:
            best_composite = composite
            best_metric    = composite   # kept for checkpoint compat
            best_epoch     = epoch
            stale_epochs   = 0
        else:
            stale_epochs += 1

        ckpt_path = save_checkpoint(
            epoch=epoch,
            model=model,
            optimizer_a=optimizer_a,     
            optimizer_b=optimizer_b,   
            scheduler_a=scheduler_a, 
            scheduler_b=scheduler_b,
            best_metric=best_metric,
            best_epoch=best_epoch,
            train_history=train_history,
            checkpoint_dir=CHECKPOINT_DIR,
            training_config=training_config,
            is_best=is_best,
        )

        epoch_bar.set_postfix({
            'train': f"{train_stats['total']:.4f}",
            'val_mae': f"{val_stats['val_mae']:.3f}",
            'val_rmse': f"{val_stats['val_rmse']:.3f}",
            'psnr': f"{val_stats['val_psnr']:.2f}",
            'best': f"{best_metric:.3f}",
            'lr_A': f"{lr_a:.2e}",
        })

        tqdm.write(
            f"Epoch {epoch:03d} | "
            f"λpin {current_lambda_pin:.2f} | "
            f"TLoss {train_stats['total']:.4f} "
            f"(B={train_stats['base']:.3f} S={train_stats['slope']:.3f} "
            f"C={train_stats['curve']:.3f} P={train_stats['pin']:.3f}) | "
            f"gA={train_stats['grad_norm_a']:.6f} gB={train_stats['grad_norm_b']:.6f} | "
            f"AnchorMAE {val_stats['val_anchor_mae']:.4f} | "
            f"RTopoFar {val_stats['val_rtopo_far']:.4f} | "
            f"ValMAE {val_stats['val_mae']:.4f} | "
            f"SlopeRMSE {val_stats['val_slope_rmse']:.4f} | "
            f"Composite {composite:.4f} | "
            f"LR (A:{lr_a:.2e} B:{lr_b:.2e}) | "
            f"Best {best_composite:.4f} @ ep {best_epoch} | "
            f"time {elapsed:.1f}s"
        )

        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            tqdm.write(f"Early stopping triggered at epoch {epoch}.")
            break

    epoch_bar.close()
    print('Training finished.')

if __name__ == '__main__':
    main()

[TRAIN DATASET] Loaded 1649 training files.
   -> Generating 2048x2048 Spatial Noise Cache in CPU RAM...
[VAL DATASET] Loaded 6 validation files.
Resumed from epoch 31 | best_composite=1.1666 @ ep 29
Train files: 1649
Val files:   6


Epochs:   0%|                                                                                  | 0/469 [00:00<…